In [2]:
import pandas as pd
import numpy as np

In [4]:
key = pd.read_csv("data/key.csv")
message = pd.read_csv("data/message.csv")
codebook = pd.read_csv("data/codebook.csv")

In [17]:
print(", ".join(v for v in key["relay"].unique()))

LaTrappe-42, TyntMeadow-88, Achel-19, Westmalle-31, Rochefort-10, Engelszell-28, Koningshoeven-05, Westvleteren-12, Chimay-23, Orval-07, TreFontane-33, MontDesCats-77, Zundert-56, Spencer-61


In [ ]:
stations = list(key["relay"].unique())
for station in stations:
    v = key[key["relay"] == station]["shift"]
    t = pd.to_numeric(v.dropna(), errors="coerce").isna().sum()
    if v.isna().mean() == 1.0:
        print(f"empty {station}")
    elif v.nunique() == 1.0:
        print(f"one unique {station}")
        print(list(v)[0])
    elif v.nunique() in [3, 4]:
        print(f"cycle {station}")
        print(v.nunique())
    elif t > 0:
        print(f"trash {station}")
        print(t)
TRASH = ["Achel-19", "Orval-07", "MontDesCats-77", "Spencer-61"]


trash Achel-19
26
cycle Orval-07
4
one unique MontDesCats-77
176
empty Spencer-61


In [ ]:
N = len(codebook)
merged = pd.merge(key[~key["relay"].isin(TRASH)], message, on="pos")
merged["shift"] = merged["shift"].fillna(-1).astype(int)

mask = merged["shift"] != -1
merged.loc[mask, "code"] = (merged.loc[mask, "code"] - merged.loc[mask, "shift"]) % N
merged.loc[~mask, "code"] = -1
arr = np.array([v.to_numpy(dtype=int) for _, v in merged.groupby(by="relay")["code"]]).T

In [127]:
import hashlib

bins = []
for i in range(len(arr)):
    mask = arr[i] == -1
    m_f = np.argmax(np.bincount(arr[i][~mask]))
    bins.append(m_f)
    arr[i][mask] = m_f

shifts = (message.sort_values("pos")["code"].to_numpy(dtype=int) - np.array(bins)) % N
print(shifts.sum())
print(hashlib.sha256(",".join(map(str, shifts)).encode()).hexdigest())


16024
08a09df10bbe00b15d9fbe7f0c2937cd3e6986aa920adc7fab83d762c443ecfd


In [ ]:
d = codebook.set_index("code")["char"].to_dict()
print("".join(d[v] for v in bins))

Привет, друг! Жители TRAPPIST-1d поздравляют тебя с началом занятий. Per aspera ad astra! У тебя все получится. Код посадочной площадки: 86W-V4W-H75-LK9


In [ ]:
trash = np.argpartition((np.array(bins)[:, None] == arr).mean(0), 2)[:2]
stations = np.array([name for name, _ in merged.groupby(by="relay")["code"]])[trash]
print(stations)

['Chimay-23' 'Westmalle-31']
